> **日本語版について**: このノートブックは、[jpmorganchase/python-training](https://github.com/jpmorganchase/python-training) の英語原文を日本語に翻訳したものです（Apache License 2.0）。コードの動作は原文から変更していません。

# Web API からのデータ取得

「Web 2.0」で大きく変わったのは、外部向け API の整備です。開発者は、公開されたデータの上にツールを組み立てられるようになりました。

API の応答は、機械が読みやすい形式で返ってきます。接続の方式も決まっているため、安定してデータを取得できます。

API は数多く公開されています。利用に開発者キーの登録が必要なものもあります。たとえば [セントルイス連邦準備銀行の API](https://research.stlouisfed.org/docs/api/) です。無料で使えるものもあります。たとえば [Coindesk の API](https://www.coindesk.com/api) です。API を探すときは、[ProgrammableWeb のディレクトリ](https://www.programmableweb.com/apis/directory) が役立ちます。

## 大いなる力には、大いなる責任が伴います

プログラムから Web サイトにアクセスするときも、マナーを守って使いましょう。

ここでは、Coindesk の API でビットコインの日次価格を取得します。Coindesk の API は、他の多くの API と同じく、JSON（JavaScript Object Notation）形式でデータを返します。JSON は、そのまま DataFrame に変換できます。使うライブラリは `json` と `requests` です。

In [ ]:
import datetime
import pandas as pd
import json
import requests

coindeskURL = 'https://api.coindesk.com/v1/bpi/historical/close.json?'

# API から取得
start = datetime.date(2019, 1 ,1)
end = datetime.date(2019, 7, 1)

url = f'{coindeskURL}start={start:%Y-%m-%d}&end={end:%Y-%m-%d}'

print(f'Hitting this url:{url}')

result = requests.get(url)
result.content

もう少し詳しく見てみましょう。データ取得で押さえるポイントはいくつかあります。

API ごとに、返ってくるデータの構造は違います。まずブラウザで URL を開いてみると、返ってくる内容がそのまま確認できることが多いです。試してみましょう。

[https://api.coindesk.com/v1/bpi/historical/close.json?start=2019-01-01&end=2019-07-01](https://api.coindesk.com/v1/bpi/historical/close.json?start=2019-01-01&end=2019-07-01)


この URL では、次の結果が返ってきます。
```
{
    "bpi":
    {
        "2019-01-01":3869.47,
        "2019-01-02":3941.2167,
        ...
    },
    "disclaimer": "This data was produced from the CoinDesk Bitcoin Price Index.",
    "time":
    {
        "updated":"Jul 2, 2019 00:03:00 UTC",
        "updatedISO":"2019-07-02T00:03:00+00:00"
    }
}
```

`bpi`、`disclaimer`、`time` の3つが、参照できるデータです。ここで主に使うのは `bpi` です。

データの分け方は API ごとに違います。他の API の例は、後で出てきます。

Python で JSON の中身を見てみます。

In [ ]:
jsondata = json.loads(result.content)
jsondata

これを pandas の DataFrame に変換します。

pandas には `read_json` という関数もあります。ただし、JSON の構造によっては使いにくい場合があります。

In [ ]:
# disclaimer や time も一緒に読み込まれる
data = pd.read_json(result.content)
data

In [ ]:
data = pd.DataFrame({'Bitcoin Price Index': jsondata['bpi']})
data

In [ ]:
%matplotlib inline
data.plot()